# Tutorial: Estimating Generalised Contact Matrices with Hi-resolution BRC model

Welcome to this tutorial on estimating age-specific social contact matrices using the Hi-resolution BRC model.

## Table of Contents

1. [Background: The van de Kassteele Method](#1-background-the-van-de-kassteele-method)
   - Why estimate contact matrices?
   - The challenge
   - The van de Kassteele approach
   - Mathematical formulation
   
2. [Generating Synthetic Contact Data](#2-generating-synthetic-contact-data)
   - Load population and template data
   - Define study population
   - Generate true contact matrix
   - Generate participant and contact data
   
3. [Applying the van de Kassteele Method](#3-applying-the-van-de-kassteele-method)
   - Data preprocessing with `CoordToColumns` and `DataLoader`
   - Initialize `BRCfine` model with `vdKassteele` prior
   - Inference with SVI (and MCMC alternative)
   - Summarize results with `ModelSummariserBRC`
   - Visualize estimated contact matrices
   
4. [Model Evaluation](#4-model-evaluation)
   - Quantitative metrics
   - Interpretation
   
5. [Summary and Key Takeaways](#5-summary-and-key-takeaways)
   - What we learned
   - When to use this method
   - Extensions and next steps

In [ ]:
# Core scientific computing
import numpy as np
import pandas as pd
import jax.numpy as jnp
from jax import random

# cntmosaic utilities and data
from cntmosaic.utils import save_tutorial_figure
from cntmosaic.datasets import load_age_distribution, load_template_patterns

# Data generation
from cntmosaic.sim import ParticipantGenerator, MatrixGenerator, ContactGenerator, Subgroup

# Data preprocessing
from cntmosaic.dataloader import CoordToColumns, DataLoader, PopulationProportion

# Modeling
from cntmosaic.models import HiBRCfine
from cntmosaic.models.priors import PSpline2D

# Analysis and visualization
from cntmosaic.analysis import ModelSummariserBRC, ModelEvaluatorBRC
from cntmosaic.vis import plot_mosaic, plot_mosaic_marginal

# Visualization
import altair as alt

# Set random seed for reproducibility
np.random.seed(42)

## 2. Generating Synthetic Contact Data

Before applying the van de Kassteele method, we need data! While real contact survey data (like POLYMOD) is ideal, we'll use `cntmosaic`'s data generation tools to create realistic synthetic data. This allows us to:
- Validate our estimation method against known truth
- Understand model performance under controlled conditions
- Experiment with different sample sizes and contact patterns

The data generation process has three steps:
1. **Define population**: Age distribution and contact characteristics
2. **Generate contact matrix**: Realistic age-age contact patterns  
3. **Simulate survey data**: Sample participants and their contacts

### 2.1 Load Population and Template Data

We'll use United States demographic data and empirically-derived contact pattern templates:

In [ ]:
templates = load_template_patterns(country='United_States', max_age=80)

df_age_dist = load_age_distribution(country='United_States', max_age=80)

age_dist = df_age_dist.P.values

### 2.2 Define Study Population

The `Subgroup` class defines a population cohort with specific characteristics:

In [ ]:
N_GRPS = 3

subgroups = [
    Subgroup(
        n=1500 // N_GRPS,                     # Placeholder, will be set later
        age_dist=age_dist // N_GRPS,      # Follow US age distribution
        mean_cint_margin=mean,  # Mean contacts per person per day
        label=label             # Population label
    )
    for mean, label in zip([5.0, 10.0, 15.0], ['low', 'mid', 'high'])
]

print(f"Defined {N_GRPS} subgroups with varying contact intensities.")

### 2.3 Generate True Contact Matrix

The `MatrixGenerator` uses template patterns to create realistic contact intensity matrices:

In [ ]:
# Initialize matrix generator with templates
matrix_gen = MatrixGenerator(templates)

# Generate a realistic contact intensity matrix
contact_matrices = matrix_gen.generate_partial(subgroups, seed=42)

for label, matrix in contact_matrices.items():
  print(f"Subgroup '{label}' mean contacts per person: {matrix.sum(axis=-1).mean():.2f}")

Let's visualize the true contact intensity matrix we'll try to recover:

In [ ]:
charts = []
for label, contact_matrix in contact_matrices.items():
  chart = plot_mosaic(contact_matrix, title=f'Contact Intensity Matrix - {label.capitalize()} subgroup', zlabel='Intensity')
  charts.append(chart)

chart_true = alt.hconcat(*charts).resolve_scale(color='independent')
chart_true

### 2.4 Generate Participant Data

The `ParticipantGenerator` creates a dataframe of survey participants with their ages:

In [ ]:
# Initialize participant generator
part_gen = ParticipantGenerator(subgroups)

# Generate participants
df_part = part_gen.generate(seed=42)

print(f"Number of participants: {len(df_part)}")
print(f"\nFirst few participants:")
print(df_part.head())

### 2.5 Generate Contact Data

The `ContactGenerator` simulates individual contacts based on the true contact matrix:

In [ ]:
cnt_gen = ContactGenerator(
    df_part=df_part,
    cint_matrices=contact_matrices,
    model='poisson'  # Contact counts follow Poisson distribution
)

# Generate contact data
df_cnt = cnt_gen.generate(seed=42)

print(f"Total contacts generated: {len(df_cnt)}")
print(f"Average contacts per participant: {len(df_cnt) / len(df_part):.2f}")
print(f"\nFirst few contacts:")
print(df_cnt.head())

### 2.6 Empirical Contact Matrix

Let's visualize the empirical (raw) contact matrix from our survey data before any modeling:

In [ ]:
emp_cint_matrices = cnt_gen.get_contact_matrix_empirical(df_cnt, normalize=True)

charts_emp = []
# Sort by the desired order: low, mid, high
for label in ['low', 'mid', 'high']:
  emp_cint_matrix = emp_cint_matrices[label]
  print(f"Empirical matrix for subgroup '{label}': mean contacts per person: {emp_cint_matrix.sum(axis=-1).mean():.2f}")
  chart_emp = plot_mosaic(emp_cint_matrix, title=f'Empirical Contact Matrix - {label.capitalize()} subgroup', zlabel='Intensity')
  charts_emp.append(chart_emp)

chart_emp = alt.hconcat(*charts_emp).resolve_scale(color='independent')
chart_emp

In [ ]:
# Define column mappings using CoordToColumns
coord_to_cols = CoordToColumns(
    age_part='age_group',      # Participant age column name
    age_cnt='age_cnt',         # Contact age column name  
    grp_vars_part='subgroup',  # Subgroup variable in participant dataframe
    id_var='id',               # Participant ID column
    age_pop='age',             # Age column in population dataframe
    size_pop='P'               # Population proportion column
)

In [ ]:
df_pop_low = df_age_dist.copy()
df_pop_low['P'] = df_pop_low['P'] / 3  # Adjust
df_pop_low['subgroup'] = 'low'

df_pop_mid = df_age_dist.copy()
df_pop_mid['P'] = df_pop_mid['P'] / 3  # Adjust
df_pop_mid['subgroup'] = 'mid'

df_pop_high = df_age_dist.copy()
df_pop_high['P'] = df_pop_high['P'] / 3  # Adjust
df_pop_high['subgroup'] = 'high'

df_subgrp_pop = pd.concat([df_pop_low, df_pop_mid, df_pop_high], ignore_index=True)

pop_props = PopulationProportion.from_counts(
  df_subgrp_pop,
  age_col='age',
  count_col='P',
  stratify_by='subgroup'
)

Now create the `DataLoader` to process and validate the data:

In [ ]:
df_part["subgroup"] = pd.Categorical(df_part["subgroup"], categories=['low', 'mid', 'high'])

# Step 3: Initialize DataLoader
dataloader = DataLoader(
    col_map=coord_to_cols,
    part=df_part,
    cnt=df_cnt,
    pop=df_age_dist,
    population_proportions=[pop_props]
)

# The DataLoader automatically:
# - Validates data consistency
# - Creates necessary indices
# - Prepares arrays for JAX/NumPyro
# - Handles population weighting

print("DataLoader initialized successfully!")

#### What does DataLoader do?

The `DataLoader` is a crucial component that:

1. **Validates consistency** between participant, contact, and population data
2. **Creates index arrays** for efficient computation with JAX
3. **Handles missing data** and edge cases
4. **Prepares offsets** including population proportions $\log(P_b)$ and sample sizes $\log(N_i)$

These preprocessed arrays are then passed directly to the NumPyro model for inference.

### 3.2 Initialize the BRCfine Model with vdKassteele Prior

The `BRCfine` class implements the **Bayesian Rate Consistency** model for fine-grained (single-year) age resolution. We combine it with the `vdKassteele` prior to get IGMRF smoothing.

#### Define the vdKassteele prior

In [ ]:
prior = {
  "rate": PSpline2D(
    prior_type='global',    # Symmetric contact matrix
    order=2,                # Second-order differences (smooth curvature)
  ),
  "subgroup": PSpline2D(
    prior_type="partial",
    order=2,
    transform="ilr"
  )
}

print("Prior configured:")

#### Understanding Prior Hyperparameters

**`prior_type='global'`**: Enforces symmetric contact matrix where $C_{a,b} = C_{b,a}$. This is appropriate when we believe contact patterns are reciprocal (contacts from age $a$ to $b$ similar to $b$ to $a$).

**`order=2`**: Uses second-order differences. This penalizes the second derivative (curvature), producing smoother surfaces than first-order. Think of it as preferring "gently rolling hills" over "sharp peaks and valleys."

**`tau_shape` and `tau_rate`**: Control the precision parameter $\tau$:
- Higher $\tau$ → more smoothing (trust prior more)
- Lower $\tau$ → less smoothing (trust data more)
- The Gamma prior allows the data to learn the optimal smoothness level
- Prior mean: $E[\tau] = \text{shape}/\text{rate} = 2.0/0.01 = 200$

#### Initialize the BRCfine model


In [ ]:
model = HiBRCfine(
    dataloader=dataloader,
    priors=prior,
    likelihood='poisson'  # Contact counts follow Poisson distribution
)

print("BRCfine model initialized:")
print(f"  Likelihood: {model.likelihood}")
print(f"  Prior: {type(prior["rate"]).__name__}")
print(f"  Partial prior for subgroup: {type(prior["subgroup"]).__name__}")

model.print_model_shape()

### 3.3 Inference with Stochastic Variational Inference (SVI)

We'll use **Stochastic Variational Inference (SVI)**, which:
- Is much faster than MCMC (minutes vs hours)
- Scales well to large datasets
- Provides approximate posterior distributions
- May underestimate uncertainty compared to exact MCMC

### Define the variational guide (mean-field normal approximation)

For SVI, we need to specify a **guide** (variational family). We'll use an autoguide that automatically constructs a mean-field approximation.

In [ ]:
from numpyro.infer.autoguide import AutoNormal
from numpyro.infer.initialization import init_to_mean

guide = AutoNormal(model.model, init_loc_fn=init_to_mean)

### Run SVI inference

In [ ]:
print("Running SVI inference...")
print("This may take a few minutes...")

prng_key = random.PRNGKey(42)

model.run_inference_svi(
    prng_key=prng_key,
    guide=guide,
    num_steps=20_000,        # Number of optimization steps
    peak_lr=0.01
)

print("Inference complete!")

#### What just happened?

During SVI, the algorithm:
1. **Initialized** variational parameters randomly
2. **Iteratively optimized** the Evidence Lower Bound (ELBO):
   $$\text{ELBO} = \mathbb{E}_{q}[\log p(\mathbf{y}, \boldsymbol{\theta})] - \mathbb{E}_{q}[\log q(\boldsymbol{\theta})]$$
3. **Learned** approximate posterior $q(\boldsymbol{\theta})$ that's close to true posterior $p(\boldsymbol{\theta}|\mathbf{y})$

The loss you see decreasing is $-\text{ELBO}$. When it plateaus, we've found our approximation!

#### Side Note: Full Bayesian Inference with MCMC

For more accurate uncertainty quantification, you can use Markov Chain Monte Carlo (MCMC) instead of SVI:

```python
# Alternative: Use MCMC for exact posterior (slower but more accurate)
model.run_inference_mcmc(
    rng_key=random.PRNGKey(42),
    num_warmup=1000,      # Burnin period
    num_samples=2000,     # Posterior samples
    num_chains=4,         # Parallel chains
    progress_bar=True
)

# Get MCMC samples
posterior_samples = model.get_posterior_samples_mcmc()
```

**Trade-offs:**
- **MCMC**: Exact posterior, better uncertainty, but 10-100x slower
- **SVI**: Fast approximate posterior, slight uncertainty underestimation

For this tutorial, SVI is sufficient and much faster!

### 3.4 Summarize Results with ModelSummariserBRC

The `ModelSummariserBRC` class processes posterior samples to compute summary statistics (median, credible intervals, etc.) for the contact matrix.

In [ ]:
# Step 9: Initialize summariser
summariser = ModelSummariserBRC(model)

# Step 10: Compute summary statistics for contact intensity matrix
summary_cint = summariser.summarise_cint(alpha=0.05) # 95% credible intervals

#### What's in the summary?

The summariser provides:
- **`median`**: Posterior median contact intensity matrix
- **`mean`**: Posterior mean (alternative point estimate)
- **`lower`**: Lower bound of 95% credible interval
- **`upper`**: Upper bound of 95% credible interval
- **`std`**: Posterior standard deviation (uncertainty measure)

These give us both point estimates and uncertainty quantification!

### 3.5 Visualize Estimated Contact Matrix

Let's visualize our estimated contact intensity matrix and compare it with the truth!

In [ ]:
# Plot estimated contact intensity (median)
chart_estimated = []
# Sort by the desired order: low, mid, high
for label in ['low', 'mid', 'high']:
  matrix = summary_cint['subgroup'][label]
  chart = plot_mosaic(
      matrix[1],
      title=f'Estimated Contact Intensity Matrix - {label.capitalize()} subgroup',
      zlabel='Intensity'
  )
  chart_estimated.append(chart)

chart_estimated = alt.hconcat(*chart_estimated).resolve_scale(color='independent')
save_tutorial_figure(chart_estimated, "HiBRC_partial_estimated_matrices", format="svg")
chart_estimated

#### Compare: True vs Estimated vs Empirical

Let's put all three side by side:

In [ ]:
chart_true & chart_emp & chart_estimated

In [ ]:
summary_mcint = summariser.summarise_mcint(alpha=0.05)

dfs = []
for label in ['low', 'mid', 'high']:
  mcint_summary = summary_mcint['subgroup'][label]
  df_mcint = pd.DataFrame({
      'age': np.arange(mcint_summary[1].shape[0]),
      'subgroup': label,
      'mid': mcint_summary[1],
      'lower': mcint_summary[0],
      'upper': mcint_summary[2]
  })
  dfs.append(df_mcint)
df_mcint_all = pd.concat(dfs, ignore_index=True)
  
lines = alt.Chart(df_mcint_all).mark_line().encode(
    x='age',
    y='mid',
    color='subgroup'
).properties(
    title='Estimated Mean Contacts per Person by Age and Subgroup'
)

bands = alt.Chart(df_mcint_all).mark_area(opacity=0.3).encode(
    x='age',
    y='lower',
    y2='upper',
    color='subgroup'
)

chart = alt.layer(lines, bands).properties(
    title='Estimated Mean Contacts per Person by Age and Subgroup'
)

chart

**Observations:**
- ✓ The van de Kassteele estimate is much smoother than the raw empirical matrix
- ✓ It captures the key features of the true matrix (diagonal pattern, age-assortative mixing)
- ✓ The IGMRF prior successfully "denoised" the data while preserving structure

### 3.6 Visualize Uncertainty

Let's examine the uncertainty in our estimates by plotting credible intervals:

uncertainty_chart = alt.hconcat(
    plot_mosaic(summary_cint['lower'], title='Lower 95% CI', zlabel='Intensity').properties(width=250, height=250),
    plot_mosaic(summary_cint['median'], title='Median Estimate', zlabel='Intensity').properties(width=250, height=250),
    plot_mosaic(summary_cint['upper'], title='Upper 95% CI', zlabel='Intensity').properties(width=250, height=250)
).resolve_scale(color='shared')

save_tutorial_figure(uncertainty_chart, "vdK_uncertainty", format="svg")
uncertainty_chart

The credible intervals show regions where we're more or less confident. Wider intervals occur in:
- Older ages (less data)
- Off-diagonal cells (fewer cross-age contacts)

---

## 4. Model Evaluation

Now let's quantitatively evaluate how well our method recovered the true contact matrix using the `ModelEvaluatorBRC` class.

In [ ]:
matrices_true = {
  "subgroup": contact_matrices
}

In [ ]:
# Step 11: Initialize evaluator with true contact matrix
evaluator = ModelEvaluatorBRC(
    summariser=summariser,
    cint_matrix_true=matrices_true  # Ground truth for comparison
)

print("Evaluator initialized with ground truth matrix")

In [ ]:
# Step 12: Compute evaluation metrics
metrics = evaluator.evaluate()

print("Model Evaluation Metrics:")
metrics


#### Interpretation of Metrics

The evaluator computes several accuracy metrics:

- **MAE (Mean Absolute Error)**: Average absolute difference between estimated and true values. Lower is better.
- **RMSE (Root Mean Square Error)**: Square root of average squared differences. Penalizes large errors more. Lower is better.
- **MAPE (Mean Absolute Percentage Error)**: Average percentage error. Lower is better.
- **R² (R-squared)**: Proportion of variance explained. Higher is better (max = 1.0).
- **Correlation**: Linear correlation between true and estimated matrices. Higher is better (max = 1.0).

These metrics quantify how well our van de Kassteele method recovered the true contact patterns!

---

## 5. Summary and Key Takeaways

### What We've Learned

In this tutorial, we successfully applied the van de Kassteele method to estimate social contact matrices from survey data. Here are the key steps:

1. **Generated realistic synthetic data** using `cntmosaic`'s simulation tools
2. **Preprocessed data** using `CoordToColumns` and `DataLoader`
3. **Specified a Bayesian model** with `BRCfine` and `vdKassteele` IGMRF prior
4. **Performed inference** using fast SVI (with MCMC alternative noted)
5. **Summarized results** with credible intervals using `ModelSummariserBRC`
6. **Evaluated accuracy** against ground truth using `ModelEvaluatorBRC`

### Key Advantages of the van de Kassteele Approach

✅ **Spatial smoothness**: IGMRF prior borrows strength across neighboring ages  
✅ **Automatic smoothing**: Data determines optimal smoothness via precision parameter $\tau$  
✅ **Uncertainty quantification**: Full Bayesian framework provides credible intervals  
✅ **No basis function selection**: Unlike splines, no arbitrary choices needed  
✅ **Computationally efficient**: Sparse precision matrix enables fast computation

### When to Use This Method

The van de Kassteele method is ideal when:
- You have individual-level contact survey data
- You want single-year age resolution
- You believe contact patterns are smooth across ages
- You need uncertainty quantification
- You have computational resources for Bayesian inference

### Extensions and Next Steps

Consider exploring:
- **Stratified models**: Estimate separate matrices by gender, location, or setting
- **Hierarchical models**: Share information across multiple surveys or countries
- **Alternative priors**: Try different smoothness assumptions (e.g., B-splines)
- **Model comparison**: Compare IGMRF vs other approaches using cross-validation

### References

**van de Kassteele, J., van Eijkeren, J., & Wallinga, J. (2017).** *Efficient estimation of age-specific social contact rates between men and women.* Annals of Applied Statistics, 11(1), 320-339.

**Mossong, J., et al. (2008).** *Social contacts and mixing patterns relevant to the spread of infectious diseases.* PLoS Medicine, 5(3), e74.

---

## 6. Additional Resources

- **cntmosaic Documentation**: [Package documentation and API reference]
- **NumPyro**: [Probabilistic programming in JAX](http://num.pyro.ai/)
- **POLYMOD Study**: Original European contact survey data
- **socialmixr R Package**: Alternative implementation in R

---

Thank you for completing this tutorial! You now have the tools to estimate sophisticated age-specific contact matrices using modern Bayesian methods. Happy modeling! 🎉